In [ ]:
# 1. Setup e bibliotecas
import os
import json
import numpy as np
import pandas as pd
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go

from src.io import paths
from src.validation import merges
from src.utils.logger import log_result

DATA_PATHS = paths.get_data_paths()
PROC_PATH = str(DATA_PATHS["data_processed"])

paths_map = {
    "labels": os.path.join(PROC_PATH, "labels_y_daily.csv"),
    "oof": os.path.join(PROC_PATH, "16_oof_predictions.csv"),
    "results16": os.path.join(PROC_PATH, "results_16_models_tfidf.json"),
}

for key, path_str in paths_map.items():
    if not os.path.exists(path_str):
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado: {path_str}")

print(json.dumps(paths_map, indent=2, ensure_ascii=False))


In [ ]:
# 2. Carregar dados e preparar dataframe base
labels = pd.read_csv(paths_map["labels"], parse_dates=["day"])
oof = pd.read_csv(paths_map["oof"], parse_dates=["day"])

if oof.empty:
    raise ValueError("Arquivo 16_oof_predictions.csv está vazio. Rode o notebook 16 primeiro.")

df = oof.merge(
    labels.drop(columns=["y"], errors="ignore"),
    on="day",
    how="left",
    suffixes=("", "_label"),
)

if "ret_next" not in df.columns:
    raise ValueError("Dados não possuem coluna ret_next para validação.")

df["sentiment"] = df["proba"] * 2 - 1

try:
    df["signal_bucket"] = pd.qcut(
        df["sentiment"],
        q=[0, 0.33, 0.66, 1],
        labels=["negativo", "neutro", "positivo"],
        duplicates="drop",
    )
except ValueError:
    df["signal_bucket"] = pd.cut(
        df["sentiment"], bins=3, labels=["negativo", "neutro", "positivo"], include_lowest=True
    )

df.sort_values(["model", "day"], inplace=True)
merges.check_intersection(oof, labels, col_left="day", col_right="day", min_days=90)

display(df.head())
print(df["model"].value_counts())


In [ ]:
# 3. Correlações Pearson/Spearman por modelo
corr_rows = []
for model_name, sub in df.groupby("model"):
    sub = sub.dropna(subset=["sentiment", "ret_next"])
    if len(sub) < 3:
        continue
    try:
        pear = stats.pearsonr(sub["sentiment"], sub["ret_next"])
    except Exception:
        pear = (np.nan, np.nan)
    try:
        spear = stats.spearmanr(sub["sentiment"], sub["ret_next"])
    except Exception:
        spear = (np.nan, np.nan)
    corr_rows.append({
        "model": model_name,
        "n": len(sub),
        "pearson_r": pear[0],
        "pearson_p": pear[1],
        "spearman_r": spear[0],
        "spearman_p": spear[1],
    })

corr_df = pd.DataFrame(corr_rows)
display(corr_df)

In [ ]:
# 4. ANOVA com buckets de sentimento
anova_rows = []
for model_name, sub in df.groupby("model"):
    valid = sub.dropna(subset=["signal_bucket", "ret_next"])
    groups = [grp["ret_next"].values for _, grp in valid.groupby("signal_bucket") if len(grp) > 0]
    if len(groups) < 2:
        continue
    try:
        f_stat, p_val = stats.f_oneway(*groups)
    except Exception:
        f_stat, p_val = np.nan, np.nan
    means = valid.groupby("signal_bucket")["ret_next"].mean().to_dict()
    anova_rows.append({
        "model": model_name,
        "f_stat": f_stat,
        "p_value": p_val,
        "mean_neg": means.get("negativo", np.nan),
        "mean_neu": means.get("neutro", np.nan),
        "mean_pos": means.get("positivo", np.nan),
        "n_obs": len(valid),
    })

anova_df = pd.DataFrame(anova_rows)
display(anova_df)

if "corr_df" in globals():
    anova_lookup = anova_df.set_index("model").to_dict("index") if not anova_df.empty else {}
    for _, row in corr_df.iterrows():
        model = row.get("model")
        metrics = {"pearson_r": float(row.get("pearson_r", 0) or 0), "spearman_r": float(row.get("spearman_r", 0) or 0)}
        extra = {
            "dataset": "sentiment_validation",
            "n_obs": int(row.get("n", 0) or 0),
            "anova_p": float((anova_lookup.get(model) or {}).get("p_value")) if anova_lookup else None,
        }
        log_result(model_name=model, dataset_name="sentiment_validation", metrics=metrics, extra=extra)

In [ ]:
# 5. Correlações com defasagens (lag -2 a +2)
lag_rows = []
for model_name, sub in df.groupby("model"):
    ordered = sub.sort_values("day").set_index("day")
    for lag in range(-2, 3):
        shifted = ordered["sentiment"].shift(lag)
        aligned = pd.concat([ordered["ret_next"], shifted], axis=1).dropna()
        if len(aligned) < 4:
            corr = np.nan
            p_val = np.nan
        else:
            try:
                corr, p_val = stats.pearsonr(aligned["sentiment"], aligned["ret_next"])
            except Exception:
                corr, p_val = np.nan, np.nan
        lag_rows.append({
            "model": model_name,
            "lag": lag,
            "corr": corr,
            "p_value": p_val,
            "n": len(aligned),
        })

lag_df = pd.DataFrame(lag_rows)
lag_pivot = lag_df.pivot(index="model", columns="lag", values="corr")
display(lag_df.head())

In [ ]:
# 6. Séries temporais e gráficos interativos
plots = {}

scatter_fig = px.scatter(
    df,
    x="sentiment",
    y="ret_next",
    color="model",
    trendline="ols",
    hover_data={"day": True, "signal_bucket": True},
    title="Sentimento (prob) vs Retorno D+1",
)
plots["scatter"] = scatter_fig
scatter_fig.show()

heatmap_fig = px.imshow(
    lag_pivot,
    labels=dict(x="Lag (dias)", y="Modelo", color="Correlação"),
    title="Correlação sentimento x retorno por defasagem",
    aspect="auto",
    text_auto=".2f",
)
plots["heatmap"] = heatmap_fig
heatmap_fig.show()

rolling_fig = go.Figure()
for model_name, sub in df.groupby("model"):
    ordered = sub.sort_values("day")
    ordered["sentiment_roll"] = ordered["sentiment"].rolling(5).mean()
    ordered["ret_roll"] = ordered["ret_next"].rolling(5).mean()
    rolling_fig.add_trace(
        go.Scatter(
            x=ordered["day"],
            y=ordered["sentiment_roll"],
            mode="lines",
            name=f"{model_name} - sentimento (média 5d)",
        )
    )
    rolling_fig.add_trace(
        go.Scatter(
            x=ordered["day"],
            y=ordered["ret_roll"],
            mode="lines",
            name=f"{model_name} - retorno (média 5d)",
            line=dict(dash="dash"),
            yaxis="y2",
        )
    )
rolling_fig.update_layout(
    title="Rolling 5d: sentimento x retorno",
    yaxis=dict(title="sentimento (z-score)"),
    yaxis2=dict(title="retorno", overlaying="y", side="right"),
)
plots["rolling"] = rolling_fig
rolling_fig.show()

In [ ]:
# 7. Salvar tabelas e HTML dos gráficos
corr_file = os.path.join(PROC_PATH, "17_sentiment_correlations.csv")
anova_file = os.path.join(PROC_PATH, "17_sentiment_anova.csv")
lag_file = os.path.join(PROC_PATH, "17_sentiment_lags.csv")
plot_file = os.path.join(PROC_PATH, "17_sentiment_dashboard.html")

corr_df.to_csv(corr_file, index=False, encoding="utf-8")
anova_df.to_csv(anova_file, index=False, encoding="utf-8")
lag_df.to_csv(lag_file, index=False, encoding="utf-8")

with open(plot_file, "w", encoding="utf-8") as f:
    for name, fig in plots.items():
        f.write(f"<h2>{name}</h2>\n")
        f.write(fig.to_html(full_html=False, include_plotlyjs="cdn"))

print("Arquivos salvos:")
print(corr_file)
print(anova_file)
print(lag_file)
print(plot_file)


In [ ]:
# 8. Resumo final
from IPython.display import Markdown

md = "**17_sentiment_validation concluido**\n"
md += f"- Modelos avaliados: {', '.join(sorted(df['model'].unique()))}\n"
md += "- Artefatos:\n"
md += "  - 17_sentiment_correlations.csv\n"
md += "  - 17_sentiment_anova.csv\n"
md += "  - 17_sentiment_lags.csv\n"
md += "  - 17_sentiment_dashboard.html\n"
md += "\n**Proximo:** notebook 18_backtest_simulation -> testar regras de trading com as probabilidades do notebook 16."

Markdown(md)
